# 01 Verify law + weights (re-fetch live)
Educational only. No personal data. Re-fetches myFICO + Experian + FTC sources live and asserts ground truth.
Run: `jupyter nbconvert --to notebook --execute 01_verify_law_and_weights.ipynb --output 01.executed.ipynb`

In [1]:
import sys; sys.path.insert(0, "../src")
import requests, re, json
UA={"User-Agent":"credit-truth-benchmark/0.1 (educational reproduction)"}
urls={
 "myfico":"https://www.myfico.com/credit-education/whats-in-your-credit-score",
 "experian609":"https://www.experian.com/blogs/ask-experian/what-is-a-609-dispute-letter/",
}
pages={}
for k,u in urls.items():
    r=requests.get(u, timeout=30, headers=UA)
    print(k, r.status_code, len(r.text))
    assert r.status_code==200, (k, r.status_code)
    pages[k]=r.text
print("fetched", list(pages.keys()))

myfico 200 214750


experian609 200 222679
fetched ['myfico', 'experian609']


In [2]:
# Assert FICO weights 35/30/15/10/10 present on myFICO page
t=pages["myfico"]
for needle in ["35%", "30%", "15%", "10%"]:
    assert needle in t, needle
    print("found", needle)
assert "payment history" in t.lower()
print("myFICO weights CONFIRMED: 35/30/15/10/10")

found 35%
found 30%
found 15%
found 10%
myFICO weights CONFIRMED: 35/30/15/10/10


In [3]:
# Assert Experian 609 = disclosure, not dispute; 611 = dispute
e=pages["experian609"].lower()
assert "isn't actually" in e or "is not" in e or "not a dispute" in e.replace("\u2019","'"), "609-disclaimer missing"
assert "section 611" in e, "611 reference missing"
assert "30 days" in e, "30-day window missing"
print("Experian 609 ground truth CONFIRMED: 609=disclosure, 611=dispute, 30d investigation")
from credit_truth.data_loaders import get_public_aggregates
print(json.dumps(get_public_aggregates()["fico_weights"], indent=2))

Experian 609 ground truth CONFIRMED: 609=disclosure, 611=dispute, 30d investigation
{
  "payment": 0.35,
  "amounts_owed": 0.3,
  "length": 0.15,
  "new_credit": 0.1,
  "mix": 0.1,
  "source": "https://www.myfico.com/credit-education/whats-in-your-credit-score",
  "retrieved": "2026-09-30"
}


In [4]:
# Cross-check simulator direction vs public aggregates
from credit_truth import fico_simulator as sim
clean=sim.score(sim.Profile())
late=sim.score(sim.Profile(n30=1, months_since_last_late=1))
hi=sim.score(sim.Profile(agg_util=0.65, max_indiv_util=0.85))
lo=sim.score(sim.Profile(agg_util=0.08, max_indiv_util=0.10))
print(f"clean={clean} late={late} drop={clean-late} (expect 60-110)")
print(f"high-util={hi} low-util={lo} gain={lo-hi} (expect 30-80)")
assert 50 <= clean-late <= 120
assert 30 <= lo-hi <= 120
print("simulator direction OK")

clean=850 late=770 drop=80 (expect 60-110)
high-util=770 low-util=850 gain=80 (expect 30-80)
simulator direction OK
